# Omni ChromHMM SAGAconf Analysis

## Compute

In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import importlib
from IPython.display import display
import glob
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

In [ ]:
# Load configuration
config_path = os.path.abspath("config_sagaconf.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(os.path.abspath(config_path))
scripts_dir = os.path.join(project_root, "scripts", "analysis")
scripts_rules_dir = os.path.join(project_root, "scripts", "rules")
scripts_root_dir = os.path.join(project_root, "scripts")
workdir = os.path.expanduser(config.get("workdir", "."))

In [ ]:
sys.path.insert(0, scripts_dir)
sys.path.insert(0, scripts_rules_dir)
sys.path.insert(0, scripts_root_dir)

import match
import utils
from utils import (
    CHROMHMM, CHROMHMM_DEFAULT,
    COSINE, COSINE_DISPLAY,
    HOMER, HOMER_DISPLAY, JACCARD, JACCARD_DISPLAY,
    JOINT_CHROMHMM, JOINT_KMEANS_HOMER, JOINT_KMEANS_MACS2, JOINT_KMEANS_OMNI,
    KAPPA, KAPPA_DISPLAY, KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI,
    MACS2, MACS2_DISPLAY, NOQH, NOQH_TYPES, OMNI, OMNI_DISPLAY,
    QUIESCENT
)
import analyze
import compare
import compare_methods
import summary_plots
import interpretation
import display as _display_helpers

# Re-import the analysis modules
for _m in (analyze, match, compare, compare_methods, summary_plots, utils,
           interpretation, _display_helpers):
    importlib.reload(_m)

# Bind the display helpers after the reload above so edits to display.py are
# picked up too. Imported as functions, not as the module: the bare name
# `display` belongs to IPython's own display().
from display import header, method_plot, show_all, show_group

In [ ]:
method_palette = {utils.display_name(m): utils.method_color(m) for m in [
    CHROMHMM_DEFAULT, KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI,
    JOINT_CHROMHMM,
    JOINT_KMEANS_HOMER, JOINT_KMEANS_MACS2, JOINT_KMEANS_OMNI,
]}

os.chdir(workdir)
os.makedirs("out/summary_plots", exist_ok=True)
print(f"Project root: {project_root}")
print(f"Workdir     : {workdir}")

P = config["params"]
DATASETS = config["datasets"]
NSTATES = P["n_states"]
MATCH_METHOD = "matched"
DO_REPLICATES = P.get("replicates", False)

METHOD_LABELS = [
    (CHROMHMM_DEFAULT, utils.display_name(CHROMHMM_DEFAULT)),
    (KMEANS_HOMER, utils.display_name(KMEANS_HOMER)),
    (KMEANS_MACS2, utils.display_name(KMEANS_MACS2)),
    (KMEANS_OMNI, utils.display_name(KMEANS_OMNI)),
    (JOINT_CHROMHMM, utils.display_name(JOINT_CHROMHMM)),
    (JOINT_KMEANS_HOMER, utils.display_name(JOINT_KMEANS_HOMER)),
    (JOINT_KMEANS_MACS2, utils.display_name(JOINT_KMEANS_MACS2)),
    (JOINT_KMEANS_OMNI, utils.display_name(JOINT_KMEANS_OMNI)),
]
METHOD_ORDER = [m[1] for m in METHOD_LABELS]
METHOD_MAP = dict(METHOD_LABELS)

def get_method_path(ds, method, rep=None):
    folder = ds
    if rep:
        folder = f"{ds}/{rep}"

    if method == CHROMHMM_DEFAULT:
        cell = DATASETS[ds]["cell"]
        return f"{folder}/{CHROMHMM_DEFAULT}_result/{cell}_{NSTATES}_dense_matched.bed"
    if method == JOINT_CHROMHMM:
        return f"{ds}/{JOINT_CHROMHMM}/{rep or 'rep1'}_{NSTATES}_dense.bed"
    if method.startswith("kmeans_"):
        caller = method.replace("kmeans_", "")
        return f"{folder}/{caller}/{caller}_kmeans_states_matched.bed"
    if method.startswith("joint_kmeans_"):
        caller = method.replace("joint_kmeans_", "")
        return f"{ds}/joint_kmeans/{caller}/{rep or 'rep1'}_kmeans_joint_states.bed"
    return None

# Segmentation layout of a dataset
# get_method_path() above resolves one segmentation; these turn the (dataset,
# method, replicate) grid into the paths, bin sizes and directories the rest of
# the notebook works with, so the layout is written down exactly once.

CHROMHMM_BIN = P["chromhmm_bin"]


def method_bin_size(method_key):
    """Native bin size of a method (each peak caller keeps its own resolution)."""
    if OMNI in method_key:
        return P.get("omni_bin", 100)
    if HOMER in method_key:
        return P.get("homer_bin", 200)
    if MACS2 in method_key:
        return P.get("macs2_bin", 100)
    return CHROMHMM_BIN


def dataset_outdir(ds):
    """Directory holding the aggregated results of one dataset."""
    return f"out/{ds}/{MATCH_METHOD}"


def method_outdir(ds, method_key, rep):
    """Analysis directory of a single segmentation."""
    outdir = f"{dataset_outdir(ds)}/{method_key}"
    return outdir if rep == "rep1" else f"{outdir}_{rep}"


def dataset_reps(ds):
    """Replicates of a dataset: rep1 and rep2, or rep1 alone."""
    return ["rep1", "rep2"] if DATASETS[ds].get("replicates", True) else ["rep1"]


def dataset_segmentations(ds):
    """(method_key, method_name, rep, path) of every segmentation of ds on disk."""
    result = []
    for method_key, method_name in METHOD_LABELS:
        for rep in dataset_reps(ds):
            path = get_method_path(ds, method_key, rep)
            if path and os.path.exists(path):
                result.append((method_key, method_name, rep, path))
    return result


# Shared analysis helpers

def nonempty(df):
    """Cache guard: an empty table means the inputs were not on disk yet."""
    return not df.empty


def load_per_dataset_tsv(filename):
    """Concatenated out/<ds>/matched/<filename> over every dataset.

    Adds the Dataset column and maps each segmentation label to its display
    Method, keeping only the methods of METHOD_ORDER. Empty until run_analysis()
    below has aggregated at least one dataset.
    """
    frames = []
    for ds in DATASETS:
        path = f"{dataset_outdir(ds)}/{filename}"
        if os.path.exists(path):
            df = pd.read_csv(path, sep="\t")
            df["Dataset"] = ds
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    # compare.run_compare() labels a segmentation "{method_key}_{rep}", so the
    # replicate is stripped before the method is resolved: display_name() has
    # no entry for a joint key carrying a rep suffix and would leave it
    # unmapped, dropping every joint row in the filter below.
    labels = df["segmentation"].astype(str)
    df["Replicate"] = labels.str.extract(r"_(rep\d)$", expand=False).fillna("rep1")
    df["Method"] = (labels.str.replace(r"_rep\d$", "", regex=True)
                    .map(utils.normalize_method).map(utils.display_name))
    return df[df["Method"].isin(METHOD_ORDER)]


def load_bin_emissions(seg_path):
    """(states, marks, matrix) of a segmentation's binarized emissions, or None."""
    path = seg_path.replace(".bed", ".bin_emissions.npz")
    if not os.path.exists(path):
        return None
    data = np.load(path, allow_pickle=False)
    return ([str(s) for s in data["states"]],
            [str(m) for m in data["marks"]],
            data["mat"].astype(float))


def emission_state_mapping(from_path, to_path):
    """One-to-one state mapping from_path -> to_path, by emission cosine.

    process_sagaconf.sh matches every model to the joint model of its own
    dataset, and each of those numbers its states arbitrarily, so state "13"
    of MCF7 has nothing to do with state "13" of K562: comparing two datasets
    by state name scores near zero whatever the method. The Hungarian match on
    the binarized emission vectors puts the two state spaces back in
    correspondence first. Only the marks both models were trained on take
    part, so a dataset missing an assay does not turn every cosine into a NaN,
    and match.background_augmented(), which emission_cosine_mapping() applies
    itself, keeps the match from scattering the states that carry no marks.

    Needed for a cross-dataset pair and for nothing else. Both replicates of a
    dataset, and its individual and joint models, were matched to that
    dataset's own joint model by match.py, which pairs states by genomic
    overlap - a stronger correspondence than an emission cosine, and one that
    rematching would replace rather than improve.

    What it cannot repair is a pair of models that split the background into a
    different number of states - HeLa-S3 ChromHMM has three quiescent states to
    CD14 monocyte's two - since then no one-to-one mapping of the two state
    spaces exists to be found.
    """
    src, dst = load_bin_emissions(from_path), load_bin_emissions(to_path)
    if src is None or dst is None:
        return None
    states_src, marks_src, mat_src = src
    states_dst, marks_dst, mat_dst = dst
    shared = [m for m in marks_src if m in set(marks_dst)]
    if not shared:
        return None
    _, mapping = match.emission_cosine_mapping(
        states_src, mat_src[:, [marks_src.index(m) for m in shared]],
        states_dst, mat_dst[:, [marks_dst.index(m) for m in shared]])
    return mapping


def compute_metrics(s1_path, s2_path, rematch=False):
    """Agreement of two segmentations, None when they cannot be compared.

    The metrics themselves come from match.agreement_metrics(); a pair that
    shares no state or never overlaps is reported as None and left out of the
    plots, rather than entering them as a row of zeros.

    rematch=True relabels s2 into the state space of s1 first, for a pair that
    does not already share one - see emission_state_mapping(). A pair whose
    emissions are not on disk is reported as None rather than compared by a
    state numbering that means nothing across the two sides.
    """
    if not os.path.exists(s1_path) or not os.path.exists(s2_path):
        return None
    s1, s2 = match.load_bed(s1_path), match.load_bed(s2_path)
    if rematch:
        mapping = emission_state_mapping(s2_path, s1_path)
        if mapping is None:
            return None
        s2 = [(chrom, start, end, mapping.get(state, state), color)
              for chrom, start, end, state, color in s2]
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    # pair_overlap() puts its *work* state first, so s2 goes in as the reference
    # and the keys come out (s1_state, s2_state), matching the l1 / l2 order.
    overlap = match.pair_overlap(s2, s1)
    states = set(l1) | set(l2)
    if not states or sum(overlap.get((a, b), 0) for a in states for b in states) == 0:
        return None
    return match.agreement_metrics(overlap, l1, l2)


def agreement_rows(pairs, desc, rematch=False):
    """DataFrame of compute_metrics() over (method_name, dataset, path1, path2)."""
    rows = []
    for method_name, ds, path1, path2 in tqdm(pairs, desc=desc):
        metrics = compute_metrics(path1, path2, rematch=rematch)
        if metrics:
            rows.append({"Method": method_name, "Dataset": ds,
                         **{utils.metric_display(m): v for m, v in metrics.items()}})
    df = pd.DataFrame(rows)
    return df


## 5. General processing (emissions and enrichment)

In [ ]:
# Configuration for analysis
GENCODE_GTF = config["tools"].get("gencode_gtf")
COORDS_DIR = config["tools"].get("coords_dir")
annotations = sorted(glob.glob(os.path.join(workdir, COORDS_DIR, "*.bed.gz"))) if COORDS_DIR else []

# Bump after changing analyze.py / compare.py to invalidate every cached aggregation.
ANALYSIS_CACHE_VERSION = 1
# Produced by compare.run_compare and consumed by the plots below; a missing one
# invalidates the stamp, so deleting a result is enough to force a rerun.
COMPARE_OUTPUTS = ["segment_stats.tsv", "entropy_summary.tsv"]


def compare_signature(ds):
    """Identity of every input compare.run_compare() reads for this dataset:
    the segmentations, their emission matrices and their per-state reports."""
    items = []
    for method_key, _, rep, path in dataset_segmentations(ds):
        items.append({
            "label": f"{method_key}_{rep}",
            "bin": method_bin_size(method_key),
            "seg": utils.file_stamp(path),
            "bin_emissions": utils.file_stamp(path.replace(".bed", ".bin_emissions.npz")),
            "bw_emissions": utils.file_stamp(path.replace(".bed", ".bw_emissions.npz")),
            "report": utils.file_stamp(f"{method_outdir(ds, method_key, rep)}/report.tsv"),
        })
    return {"version": ANALYSIS_CACHE_VERSION, "skip_noqh": True, "items": items}


def run_analysis(force=False):
    """Analyze every segmentation, then aggregate the results per dataset.

    Both steps are cached, so re-running the cell only computes what is missing:
    a segmentation is skipped once its report.tsv is there, and the (expensive)
    aggregation of a dataset is skipped while its inputs are unchanged since the
    stamp of the cached run, out/<ds>/matched/compare_stamp.json.
    force=True recomputes everything.
    """
    futures = {}
    reanalyzed = set()   # datasets with a fresh analysis: aggregation must rerun
    # We use a subset of markers if necessary, but here we assume everything is set up.
    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() // 2)) as executor:
        for ds in DATASETS:
            for method_key, method_name, rep, path in dataset_segmentations(ds):
                outdir = method_outdir(ds, method_key, rep)
                if not force and os.path.exists(f"{outdir}/report.tsv"):
                    continue
                print(f"Analyzing {method_name} ({rep}) for {ds}...")

                # Determine inputs (binarized data)
                inputs = None
                if CHROMHMM in method_key:
                    inputs = [f"{ds}/{rep}/chromhmm_default/*.txt"]
                elif "kmeans" in method_key:
                    caller = method_key.replace("joint_", "").replace("kmeans_", "")
                    inputs = [f"{ds}/{rep}/{caller}/chromhmm_peaks/chr*.txt.gz"]

                reanalyzed.add(ds)
                fut = executor.submit(
                    analyze.run_analyze,
                    seg=path, bin_size=method_bin_size(method_key),
                    outdir=outdir,
                    inputs=inputs,
                    annotations=annotations,
                    bw_emissions=path.replace(".bed", ".bw_emissions.npz"),
                    emissions_only=False,
                    skip_noqh=True,
                )
                futures[fut] = f"{method_name} ({rep}) for {ds}"

    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception as e:
            print(f"  ERROR: {futures[fut]}: {e}")

    # 2. Aggregation step: run compare.run_compare to produce segment_stats.tsv and entropy_summary.tsv
    for ds in DATASETS:
        segmentations = dataset_segmentations(ds)
        if not segmentations:
            continue

        comp_outdir = dataset_outdir(ds)
        stamp_path = f"{comp_outdir}/compare_stamp.json"
        # Stamped before the run, so a segmentation that changes while compare is
        # running is not recorded as already aggregated.
        signature = compare_signature(ds)
        outputs = [f"{comp_outdir}/{name}" for name in COMPARE_OUTPUTS]
        if not force and ds not in reanalyzed and utils.stamp_current(stamp_path, signature, outputs):
            print(f"Aggregation for {ds} is up to date, skipped.")
            continue

        segs = [path for _, _, _, path in segmentations]
        bins = [method_bin_size(method_key) for method_key, _, _, _ in segmentations]
        # Important: labels must follow the _rep1 / _rep2 convention for should_compare to work
        labels = [f"{method_key}_{rep}" for method_key, _, rep, _ in segmentations]

        print(f"Aggregating results for {ds}...")
        compare.run_compare(seg=segs, bins=bins, labels=labels, all_pairs=False,
                            outdir=comp_outdir, analysis_dir=comp_outdir,
                            skip_noqh=True)
        utils.save_stamp(stamp_path, signature)


# Run analysis if needed
run_analysis()


## 0. Peak calling stats per methods

In [ ]:
def compute_peak_stats():
    frames = []
    for ds in DATASETS:
        p = f"{ds}/peaks/peak_stats.tsv"
        if os.path.exists(p):
            df = pd.read_csv(p, sep='\t')
            # Filter marks based on DATASETS[ds]['marks'] if available
            if 'marks' in DATASETS[ds]:
                df = df[df['mark'].isin(DATASETS[ds]['marks'])]
            df['Dataset'] = ds
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    df_all = pd.concat(frames, ignore_index=True)
    df_all['method'] = df_all['method'].replace('Default', utils.display_name(CHROMHMM_DEFAULT))
    df_all['method'] = df_all['method'].replace(HOMER_DISPLAY, utils.display_name(KMEANS_HOMER))
    df_all['method'] = df_all['method'].replace(MACS2_DISPLAY, utils.display_name(KMEANS_MACS2))
    df_all['method'] = df_all['method'].replace(OMNI_DISPLAY, utils.display_name(KMEANS_OMNI))
    return df_all


df_peaks = utils.cached_pickle("out/df_peaks.pkl", compute_peak_stats,
                               label="peak statistics", valid=nonempty)

if not df_peaks.empty:

    order = [
        utils.display_name(CHROMHMM_DEFAULT),
        utils.display_name(KMEANS_HOMER),
        utils.display_name(KMEANS_MACS2),
        utils.display_name(KMEANS_OMNI)
    ]
    marks_present = df_peaks['mark'].unique()
    # Canonical order from summary_plots.py
    MARK_ORDER = ["H3K4me3", "H3K27ac", "H3K4me1", "H3K36me3", "H3K9me3", "H3K27me3"]
    marks = [m for m in MARK_ORDER if m in marks_present]
    marks += sorted([m for m in marks_present if m not in MARK_ORDER])

    # Plot 1: Number of peaks
    plt.figure(figsize=(max(8, len(marks) * 0.8), 5))
    sns.barplot(data=df_peaks, x='mark', y='n_peaks', order=marks,
                hue='method', hue_order=order, palette=method_palette,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    plt.title("Number of peaks per mark", fontsize=11, fontweight="bold")
    plt.ylabel("Count", fontsize=9)
    plt.grid(axis='y', alpha=0.3)
    utils.strip_points(plt.gca(), data=df_peaks, x='mark', y='n_peaks', order=marks,
                       hue='method', hue_order=order, size=2)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.legend(title="Method", fontsize=8, title_fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("out/summary_plots/n_peaks.png", bbox_inches='tight')
    plt.close()

    # Plot 2: Average peak length
    plt.figure(figsize=(max(8, len(marks) * 0.8), 5))
    sns.barplot(data=df_peaks, x='mark', y='mean_length', order=marks,
                hue='method', hue_order=order, palette=method_palette,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    plt.title("Average peak length per mark", fontsize=11, fontweight="bold")
    plt.ylabel("Length (bp)", fontsize=9)
    plt.grid(axis='y', alpha=0.3)
    utils.strip_points(plt.gca(), data=df_peaks, x='mark', y='mean_length', order=marks,
                       hue='method', hue_order=order, size=2)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.legend(title="Method", fontsize=8, title_fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("out/summary_plots/peak_length.png", bbox_inches='tight')
    plt.close()


## 1. Number of segments stats per methods

In [ ]:
df_segs = utils.cached_pickle(
    "out/df_segs.pkl", lambda: load_per_dataset_tsv("segment_stats.tsv"),
    label="segment statistics", valid=nonempty)

if not df_segs.empty:

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_segs, x='Method', y='n_segments', order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_segs, x='Method', y='n_segments', order=METHOD_ORDER, size=2)
    plt.title("Number of segments per method", fontsize=11, fontweight="bold")
    plt.ylabel("Number of segments", fontsize=9)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_segs[df_segs["Method"] == method]["n_segments"]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{int(m)}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/n_segments.png", bbox_inches='tight')
    plt.close()


## 2. Transition matrix entropy per method

In [ ]:
df_entropy = utils.cached_pickle(
    "out/df_entropy.pkl", lambda: load_per_dataset_tsv("entropy_summary.tsv"),
    label="transition matrix entropy", valid=nonempty)

if not df_entropy.empty:

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_entropy, x='Method', y='total_entropy', order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_entropy, x='Method', y='total_entropy', order=METHOD_ORDER, size=2)
    plt.title("Transition matrix entropy per method", fontsize=11, fontweight="bold")
    plt.ylabel("Entropy (bits)", fontsize=9)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_entropy[df_entropy["Method"] == method]["total_entropy"]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/entropy.png", bbox_inches='tight')
    plt.close()


## 3. State composition and lengths

In [ ]:
# Collect valid tasks for state-based analysis
valid_tasks = [(f"{ds}_{rep}", method_name, path)
               for ds in DATASETS
               for _, method_name, rep, path in dataset_segmentations(ds)]

def state_comp_worker(task):
    folder, method_name, path = task
    try:
        segs = match.load_bed(path)
        lengths_by_state = defaultdict(list)
        for row in segs:
            name = row[3]
            if not name or name == ".": continue
            if "Unknown" in name:
                raise ValueError(f"Unknown state detected in {path}: {name}")
            lengths_by_state[name].append(row[2] - row[1])

        tot_bp = sum(sum(ls) for ls in lengths_by_state.values())

        res = []
        for st, ls in lengths_by_state.items():
            res.append({
                "Dataset": folder,
                "Method": method_name,
                "State": st,
                "Fraction": sum(ls) / tot_bp if tot_bp > 0 else 0,
                "MeanLength": np.mean(ls),
                "MedianLength": np.median(ls)
            })
        return res
    except Exception as e:
        return []

def compute_state_composition():
    """(composition table, {state: hex color}) over every segmentation."""
    per_task = [state_comp_worker(t) for t in tqdm(valid_tasks)]
    df = pd.DataFrame([row for rows in per_task for row in rows])

    # State colors, from the itemRgb column of the segmentation BEDs
    colors_hex = {}
    for _, _, path in tqdm(valid_tasks, desc="Extracting state colors"):
        if not os.path.exists(path): continue
        for row in match.load_bed(path):
            name = row[3]
            color = row[8] if len(row) > 8 else None
            if name not in colors_hex and color and color != "0,0,0":
                colors_hex[name] = analyze.rgb_str_to_hex(color)
    return df, colors_hex


df_comp, state_colors_hex = utils.cached_pickle(
    "out/df_comp.pkl", compute_state_composition,
    label="state composition and lengths")

# Fill missing colors from summary_plots.STATE_COLORS
states_present = df_comp["State"].unique()
for s in states_present:
    if s not in state_colors_hex or state_colors_hex[s] == "#000000":
        name_part = s.split("_")[1] if "_" in s else s
        found_color = None
        for canonical, rgb in summary_plots.STATE_COLORS.items():
            if name_part.startswith(canonical):
                found_color = "#{:02x}{:02x}{:02x}".format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                break
        if found_color:
            state_colors_hex[s] = found_color
        else:
            state_colors_hex[s] = "#888888"

states_order = summary_plots.sort_states(df_comp["State"].unique())
# Plot 3a. Average state composition per method (broken axis)
df_comp_filled = df_comp.pivot_table(index=["Method", "Dataset"], columns="State", values="Fraction", fill_value=0).stack().reset_index(name="Fraction")

BREAK_LOW = 0.20
BREAK_HIGH = 0.40
figw = max(12, len(states_order) * len(METHOD_ORDER) * 0.22)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(figw, 6),
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.06},
)

for ax in (ax_top, ax_bot):
    sns.barplot(
        data=df_comp_filled, x="State", y="Fraction", hue="Method",
        order=states_order, hue_order=METHOD_ORDER, palette=method_palette,
        ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
        legend=(ax is ax_top), edgecolor="lightgrey", linewidth=1
    )
    utils.hatch_joint(ax, METHOD_ORDER)
    utils.strip_points(ax, data=df_comp_filled, x="State", y="Fraction", hue="Method",
                       order=states_order, hue_order=METHOD_ORDER,
                       size=1.5, alpha=0.4, jitter=0.2)

ax_top.set_ylim(BREAK_HIGH, 1.02)
ax_bot.set_ylim(0, BREAK_LOW)
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", bottom=False)

d = 0.012
kwargs = dict(transform=fig.transFigure, color="k", clip_on=False, linewidth=0.8)
for ax, sign in [(ax_top, -1), (ax_bot, 1)]:
    x0, x1 = ax.get_position().x0, ax.get_position().x1
    y = ax.get_position().y0 if sign == 1 else ax.get_position().y1
    for x in (x0, x1):
        fig.add_artist(plt.Line2D([x - d, x + d], [y + sign * d * 1.5, y - sign * d * 1.5], **kwargs))

for ax in (ax_top, ax_bot):
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="y", labelsize=8)

ax_bot.tick_params(axis="x", labelsize=8, rotation=45)
ax_bot.set_xlabel("Chromatin state", fontsize=9)
ax_top.set_xlabel("")
ax_top.set_title("Average state composition per method", fontsize=10, fontweight="bold")
ax_top.legend(title="Method", fontsize=8, title_fontsize=9,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
if ax_bot.get_legend(): ax_bot.get_legend().remove()
ax_bot.set_ylabel("Fraction of genome", fontsize=9)
ax_top.set_ylabel("")

plt.savefig("out/summary_plots/avg_composition.png", bbox_inches="tight")
plt.close(fig)

# Plot 3b. Average state mean length per method
plt.figure(figsize=(figw, 6))
ax = plt.gca()
sns.barplot(
    data=df_comp, x="State", y="MeanLength", hue="Method",
    order=states_order, hue_order=METHOD_ORDER, palette=method_palette,
    ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, edgecolor="lightgrey", linewidth=1
)
utils.hatch_joint(ax, METHOD_ORDER)
utils.strip_points(ax, data=df_comp, x="State", y="MeanLength", hue="Method",
                   order=states_order, hue_order=METHOD_ORDER,
                   size=1.5, alpha=0.4, jitter=0.2)
if not df_comp.empty and (df_comp["MeanLength"] > 0).any():
    ax.set_yscale("log")
ax.set_title("Average state mean length per method", fontsize=10, fontweight="bold")
ax.set_ylabel("Mean length (bp, log scale)", fontsize=9)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.tick_params(axis="x", labelsize=8, rotation=45)
ax.legend(title="Method", fontsize=8, title_fontsize=9,
          bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.savefig("out/summary_plots/avg_mean_length.png", bbox_inches="tight")
plt.close()

# Plot 3c. Average state median length per method
plt.figure(figsize=(figw, 6))
ax = plt.gca()
sns.barplot(
    data=df_comp, x="State", y="MedianLength", hue="Method",
    order=states_order, hue_order=METHOD_ORDER, palette=method_palette,
    ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, edgecolor="lightgrey", linewidth=1
)
utils.hatch_joint(ax, METHOD_ORDER)
utils.strip_points(ax, data=df_comp, x="State", y="MedianLength", hue="Method",
                   order=states_order, hue_order=METHOD_ORDER,
                   size=1.5, alpha=0.4, jitter=0.2)
if not df_comp.empty and (df_comp["MedianLength"] > 0).any():
    ax.set_yscale("log")
ax.set_title("Average state median length per method", fontsize=10, fontweight="bold")
ax.set_ylabel("Median length (bp, log scale)", fontsize=9)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.tick_params(axis="x", labelsize=8, rotation=45)
ax.legend(title="Method", fontsize=8, title_fontsize=9,
          bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.savefig("out/summary_plots/avg_median_length.png", bbox_inches="tight")
plt.close()

# Plot 3d. Summary average state composition per method (stacked)
pivot_avg = df_comp.pivot_table(index=["Method", "Dataset"], columns="State", values="Fraction", fill_value=0).groupby("Method").mean()
pivot_avg = pivot_avg.reindex(METHOD_ORDER)
avail_states = [s for s in states_order if s in pivot_avg.columns]
pivot_avg = pivot_avg[avail_states]
colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

sums = pivot_avg.sum(axis=1)
if not np.allclose(sums[sums > 0], 1.0, atol=1e-5):
    raise ValueError("Average composition not normalized to 1.0")

plt.figure(figsize=(8, 5))
ax = pivot_avg.plot(kind="bar", stacked=True, color=colors, width=0.6, ax=plt.gca(), linewidth=0)
pivot_avg.sum(axis=1).plot(kind="bar", ax=ax, width=0.6, facecolor="none", edgecolor="lightgrey", linewidth=1, legend=False)
ax.set_title("Average state composition per method", fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize="small", title="State")
ax.set_xlabel("Method", fontsize=9)
ax.set_ylabel("Average Fraction of Genome", fontsize=9)
ax.tick_params(axis="x", rotation=45, labelsize=8)
for label in ax.get_xticklabels():
    label.set_ha("right")
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
plt.tight_layout()
plt.savefig("out/summary_plots/avg_composition_stacked.png", bbox_inches="tight")
plt.close()

# Plot 3e. State composition per dataset for each method
for method in METHOD_ORDER:
    method_df = df_comp[df_comp["Method"] == method]
    if method_df.empty: continue
    pivot_df = method_df.pivot(index="Dataset", columns="State", values="Fraction").fillna(0)
    avail_states = [s for s in states_order if s in pivot_df.columns]
    pivot_df = pivot_df[avail_states]
    colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

    if not np.allclose(pivot_df.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f"Composition for method {method} not normalized to 1.0")

    ax = pivot_df.plot(kind="bar", stacked=True, figsize=(max(10, len(pivot_df)*0.5), 6), color=colors, width=0.8, linewidth=0)
    pivot_df.sum(axis=1).plot(kind="bar", ax=ax, width=0.8, facecolor="none", edgecolor="lightgrey", linewidth=1, legend=False)
    ax.set_title(f"State composition per dataset - {method}", fontsize=11, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize="small", title="State")
    ax.set_xlabel("Dataset", fontsize=9)
    ax.set_ylabel("Fraction of Genome", fontsize=9)
    ax.tick_params(axis="x", labelsize=8, rotation=45)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(f"out/summary_plots/composition_{method.lower().replace(' ', '_')}.png", bbox_inches="tight")
    plt.close(fig)


## 4. Replicate consistency (Jaccard / Kappa / Cosine)

In [ ]:
def compute_replicate_agreement():
    """rep1 vs rep2 of the same method, for every dataset that has both."""
    pairs = [(method_name, ds,
              get_method_path(ds, method_key, "rep1"),
              get_method_path(ds, method_key, "rep2"))
             for ds in DATASETS
             if os.path.isdir(f"{ds}/rep1") and os.path.isdir(f"{ds}/rep2")
             for method_key, method_name in METHOD_LABELS
             if get_method_path(ds, method_key, "rep1")
             and get_method_path(ds, method_key, "rep2")]
    return agreement_rows(pairs, "Replicates")


df_rep = utils.cached_pickle("out/df_rep.pkl", compute_replicate_agreement,
                             label="replicate consistency", valid=nonempty)

if not df_rep.empty:

    # Plot 1: Jaccard
    plt.figure(figsize=(8, 6))
    sns.barplot(data=df_rep, x='Method', y=JACCARD_DISPLAY, order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_rep, x='Method', y=JACCARD_DISPLAY, order=METHOD_ORDER, size=2)
    plt.title("Replicate Jaccard Similarity", fontsize=11, fontweight="bold")
    plt.ylabel(JACCARD_DISPLAY, fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_rep[df_rep["Method"] == method][JACCARD_DISPLAY]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/replicate_jaccard.png", bbox_inches='tight')
    plt.close()

    # Plot 2: Kappa
    plt.figure(figsize=(8, 6))
    sns.barplot(data=df_rep, x='Method', y=KAPPA_DISPLAY, order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_rep, x='Method', y=KAPPA_DISPLAY, order=METHOD_ORDER, size=2)
    plt.title("Replicate Cohen's Kappa", fontsize=11, fontweight="bold")
    plt.ylabel(KAPPA_DISPLAY, fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_rep[df_rep["Method"] == method][KAPPA_DISPLAY]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/replicate_kappa.png", bbox_inches='tight')
    plt.close()

    # Plot 3: Cosine
    plt.figure(figsize=(8, 6))
    sns.barplot(data=df_rep, x='Method', y=COSINE_DISPLAY, order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_rep, x='Method', y=COSINE_DISPLAY, order=METHOD_ORDER, size=2)
    plt.title("Replicate cosine composition", fontsize=11, fontweight="bold")
    plt.ylabel(COSINE_DISPLAY, fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_rep[df_rep["Method"] == method][COSINE_DISPLAY]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/replicate_cosine.png", bbox_inches='tight')
    plt.close()


## 4. Cross-sample concordance (Jaccard / Kappa / Cosine)

In [ ]:
# The same method on two different cell lines: how much of a segmentation is
# the biology of the sample rather than the method's own signature. rep1 only,
# so the replicate variation of the section above does not enter twice, and
# with the state spaces rematched, since the two datasets were matched to
# different joint models.
def compute_cross_sample_agreement():
    """Every pair of cell lines, per method."""
    datasets = list(DATASETS)
    pairs = [(method_name, f"{ds_a}/{ds_b}",
              get_method_path(ds_a, method_key, "rep1"),
              get_method_path(ds_b, method_key, "rep1"))
             for method_key, method_name in METHOD_LABELS
             for i, ds_a in enumerate(datasets)
             for ds_b in datasets[i + 1:]
             if get_method_path(ds_a, method_key, "rep1")
             and get_method_path(ds_b, method_key, "rep1")]
    return agreement_rows(pairs, "Cross-sample", rematch=True)


df_cross = utils.cached_pickle("out/df_cross_sample.pkl",
                               compute_cross_sample_agreement,
                               label="cross-sample concordance", valid=nonempty)

if not df_cross.empty:

    for metric, title in [(JACCARD_DISPLAY, "Cross-sample Jaccard Similarity"),
                          (KAPPA_DISPLAY, "Cross-sample Cohen's Kappa"),
                          (COSINE_DISPLAY, "Cross-sample cosine composition")]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df_cross, x='Method', y=metric, order=METHOD_ORDER,
                    palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                    edgecolor="lightgrey", linewidth=1)
        utils.hatch_joint(plt.gca(), METHOD_ORDER)
        utils.strip_points(plt.gca(), data=df_cross, x='Method', y=metric, order=METHOD_ORDER, size=2)
        plt.title(title, fontsize=11, fontweight="bold")
        plt.ylabel(metric, fontsize=9)
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis='y', alpha=0.3)
        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(METHOD_ORDER):
            vals = df_cross[df_cross["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
        plt.savefig(f"out/summary_plots/cross_sample_{utils.normalize_metric(metric)}.png",
                    bbox_inches='tight')
        plt.close()


## 4. Individual vs Joint correspondence (Jaccard / Kappa)

In [ ]:
# Each peak caller, segmented on its own dataset vs segmented jointly.
COMP_PAIRS = [
    (utils.display_name(CHROMHMM_DEFAULT), CHROMHMM_DEFAULT, JOINT_CHROMHMM),
    (utils.display_name(KMEANS_HOMER), KMEANS_HOMER, JOINT_KMEANS_HOMER),
    (utils.display_name(KMEANS_MACS2), KMEANS_MACS2, JOINT_KMEANS_MACS2),
    (utils.display_name(KMEANS_OMNI), KMEANS_OMNI, JOINT_KMEANS_OMNI),
]


def compute_joint_indiv_agreement():
    pairs = [(name, ds,
              get_method_path(ds, indiv_key, "rep1"),
              get_method_path(ds, joint_key, "rep1"))
             for ds in DATASETS
             for name, indiv_key, joint_key in COMP_PAIRS
             if get_method_path(ds, indiv_key, "rep1")
             and get_method_path(ds, joint_key, "rep1")]
    return agreement_rows(pairs, "Joint vs Individual")


df_ji = utils.cached_pickle("out/df_ji.pkl", compute_joint_indiv_agreement,
                            label="joint vs individual consistency", valid=nonempty)

if not df_ji.empty:
    order = [
        utils.display_name(CHROMHMM_DEFAULT),
        utils.display_name(KMEANS_HOMER),
        utils.display_name(KMEANS_MACS2),
        utils.display_name(KMEANS_OMNI)
    ]

    # Plot 1: Jaccard
    plt.figure(figsize=(7, 6))
    sns.barplot(data=df_ji, x='Method', y=JACCARD_DISPLAY, order=order,
                palette=method_palette, hue='Method', dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.strip_points(plt.gca(), data=df_ji, x='Method', y=JACCARD_DISPLAY, order=order, size=2)
    plt.title("Individual vs Joint Jaccard", fontsize=11, fontweight="bold")
    plt.ylabel(JACCARD_DISPLAY, fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method_name in enumerate(order):
        vals = df_ji[df_ji["Method"] == method_name][JACCARD_DISPLAY]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/joint_indiv_jaccard.png", bbox_inches='tight')
    plt.close()

    # Plot 2: Kappa
    plt.figure(figsize=(7, 6))
    sns.barplot(data=df_ji, x='Method', y=KAPPA_DISPLAY, order=order,
                palette=method_palette, hue='Method', dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.strip_points(plt.gca(), data=df_ji, x='Method', y=KAPPA_DISPLAY, order=order, size=2)
    plt.title("Individual vs Joint Kappa", fontsize=11, fontweight="bold")
    plt.ylabel(KAPPA_DISPLAY, fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method_name in enumerate(order):
        vals = df_ji[df_ji["Method"] == method_name][KAPPA_DISPLAY]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/joint_indiv_kappa.png", bbox_inches='tight')
    plt.close()

    # Plot 3: Cosine
    plt.figure(figsize=(7, 6))
    sns.barplot(data=df_ji, x='Method', y=COSINE_DISPLAY, order=order,
                palette=method_palette, hue='Method', dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.strip_points(plt.gca(), data=df_ji, x='Method', y=COSINE_DISPLAY, order=order, size=2)
    plt.title("Individual vs joint cosine composition", fontsize=11, fontweight="bold")
    plt.ylabel(COSINE_DISPLAY, fontsize=9)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method_name in enumerate(order):
        vals = df_ji[df_ji["Method"] == method_name][COSINE_DISPLAY]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/joint_indiv_cosine.png", bbox_inches='tight')
    plt.close()


## 6. Chromatin state type interpretation


In [ ]:
# The interpretation itself lives in scripts/interpretation.py: it turns the
# per-state evidence of a segmentation (bin_emissions/state_emissions.tsv,
# enrichment/enrichment.tsv, report.tsv, entropy/transition_matrix.tsv) into one
# of the chromatin state types of the table shipped next to it,
# scripts/interpretation.csv.
TYPES_TABLE = interpretation.load_interpretation_table()
TYPE_ORDER = list(TYPES_TABLE.index)                    # order used in the CSV
TYPE_SHORT = TYPES_TABLE["Short Name"].to_dict()
TYPE_COLORS = interpretation.TYPE_COLORS

print(f"Chromatin state types from {interpretation.DEFAULT_INTERPRETATION_CSV}:")
display(TYPES_TABLE)
print("Marks per type family:", interpretation.MARK_GROUPS)

In [ ]:
def compute_interpretation():
    """One row per state of every segmentation analyzed by run_analysis()."""
    MATCHING_JOINT = {
        utils.display_name(CHROMHMM_DEFAULT): utils.display_name(JOINT_CHROMHMM),
        utils.display_name(KMEANS_HOMER): utils.display_name(JOINT_KMEANS_HOMER),
        utils.display_name(KMEANS_MACS2): utils.display_name(JOINT_KMEANS_MACS2),
        utils.display_name(KMEANS_OMNI): utils.display_name(JOINT_KMEANS_OMNI),
    }

    # 1. Precompute joint interpretations (once per dataset, as they are joint)
    joint_interpretations = {} # (ds, joint_method_name) -> df
    for ds in tqdm(DATASETS, desc="Joint Interpretation"):
        for method_key, method_name in METHOD_LABELS:
            if method_name in MATCHING_JOINT.values():
                # For joint models, we use rep1 to get the common interpretation
                outdir = method_outdir(ds, method_key, "rep1")
                df = interpretation.interpret_segmentation(outdir, TYPES_TABLE)
                if df is not None:
                    joint_interpretations[(ds, method_name)] = df

    # 2. Assign interpretations to all segmentations
    interpretations = []
    for ds in tqdm(DATASETS, desc="Interpretation"):
        reps = dataset_reps(ds)
        for rep in reps:
            for method_key, method_name in METHOD_LABELS:
                outdir = method_outdir(ds, method_key, rep)
                
                joint_name = MATCHING_JOINT.get(method_name)
                if joint_name:
                    # Individual method: transfer from its joint counterpart
                    joint_df = joint_interpretations.get((ds, joint_name))
                    if joint_df is None:
                        continue
                    
                    # Still need the individual report for its specific genome stats
                    ev = interpretation.load_segmentation_evidence(outdir)
                    if ev is None or ev.get("report") is None:
                        continue
                        
                    df = joint_df.copy()
                    # Preserve individual stats
                    rep_stats = ev["report"]
                    # Ensure alignment of states between individual and joint model
                    df["Fraction"] = rep_stats["total_bp"].reindex(df.index) / rep_stats["total_bp"].sum()
                    df["MedianLength"] = rep_stats["median_length"].reindex(df.index)
                    if ev.get("self_transition") is not None:
                        df["SelfTransition"] = ev["self_transition"].reindex(df.index)
                    else:
                        df["SelfTransition"] = np.nan
                else:
                    # Joint method or other: interpret normally (already computed or fresh)
                    if (ds, method_name) in joint_interpretations:
                        df = joint_interpretations[(ds, method_name)].copy()
                        # Update stats for the specific replicate if they differ
                        ev = interpretation.load_segmentation_evidence(outdir)
                        if ev is not None and ev.get("report") is not None:
                             rep_stats = ev["report"]
                             df["Fraction"] = rep_stats["total_bp"].reindex(df.index) / rep_stats["total_bp"].sum()
                             df["MedianLength"] = rep_stats["median_length"].reindex(df.index)
                             if ev.get("self_transition") is not None:
                                 df["SelfTransition"] = ev["self_transition"].reindex(df.index)
                    else:
                        df = interpretation.interpret_segmentation(outdir, TYPES_TABLE)
                
                if df is not None:
                    df = df.reset_index().rename(columns={"state": "State", "index": "State"})
                    df.insert(0, "Rep", rep)
                    df.insert(0, "Method", method_name)
                    df.insert(0, "Dataset", ds)
                    interpretations.append(df)
    return pd.concat(interpretations, ignore_index=True) if interpretations else pd.DataFrame()


df_interp = utils.cached_pickle("out/df_interp_matched.pkl", compute_interpretation,
                                label="state interpretation")

if not df_interp.empty:
    interp_tsv = "out/state_interpretation.tsv"
    df_interp.to_csv(interp_tsv, sep="\t", index=False, float_format="%.4f")
    print(f"{len(df_interp)} states of {len(df_interp.groupby(['Dataset', 'Rep', 'Method']))} "
          f"segmentations interpreted -> {interp_tsv}")

    # Interpreted type of every state, per dataset.
    for ds in DATASETS:
        for rep in ["rep1", "rep2"]:
            mat = interpretation.type_matrix(df_interp, ds, rep, methods=METHOD_ORDER)
            if mat.empty:
                continue
            print(f"\n### {ds} ({rep})")
            display(mat)

    # Types never assigned by any method — CTCF is expected here, none of these
    # datasets measures CTCF, and the table has no mark-based fallback for it.
    missing = [t for t in TYPE_ORDER if t not in set(df_interp["Type"])]
    print(f"\nTypes not assigned in any segmentation: {missing or 'none'}")


In [ ]:
if not df_interp.empty:
    seg_keys = ["Dataset", "Rep", "Method"]

    # Plot 6a. Interpreted type of every state, per dataset
    for ds in DATASETS:
        mat = interpretation.type_matrix(df_interp, ds, "rep1", column="Type",
                                         methods=METHOD_ORDER)
        if mat.empty:
            continue
        fig, ax = plt.subplots(figsize=(0.62 * mat.shape[1] + 3, 0.42 * len(mat) + 1.6))
        for i, method in enumerate(mat.index):
            for j, state in enumerate(mat.columns):
                t = mat.iat[i, j]
                if not isinstance(t, str):
                    continue
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=TYPE_COLORS[t],
                                           edgecolor="white", linewidth=1))
                ax.text(j + 0.5, i + 0.5, TYPE_SHORT[t].replace(" ", "\n"), ha="center",
                        va="center", fontsize=6, color=interpretation.type_text_color(t))
        ax.set_xlim(0, mat.shape[1])
        ax.set_ylim(len(mat), 0)
        ax.set_xticks(np.arange(mat.shape[1]) + 0.5)
        ax.set_xticklabels(mat.columns, fontsize=7)
        ax.set_yticks(np.arange(len(mat)) + 0.5)
        ax.set_yticklabels(mat.index, fontsize=8)
        ax.set_title(f"Interpreted chromatin state types - {ds} (rep1)", fontsize=10,
                     fontweight="bold")
        ax.set_xlabel("State (in the order of the segmentation)", fontsize=9)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)
        plt.tight_layout()
        plt.savefig(f"out/summary_plots/interpretation_{ds}.png", bbox_inches="tight", dpi=150)
        plt.close(fig)

    # Genome fraction per interpreted type: types absent from a segmentation
    # contribute 0, so the bars compare methods on the same footing.
    frac = (df_interp.groupby(seg_keys + ["Type"])["Fraction"].sum()
            .unstack("Type").reindex(columns=TYPE_ORDER).fillna(0.0))
    frac_long = frac.stack().reset_index().rename(columns={0: "Fraction", "level_3": "Type"})

    # Plot 6b. Genome composition by interpreted type (stacked)
    pivot_types = frac.groupby("Method").mean().reindex(METHOD_ORDER).dropna(how="all")
    fig, ax = plt.subplots(figsize=(8, 5))
    pivot_types.plot(kind="bar", stacked=True, ax=ax, width=0.6, linewidth=0,
                     color=[TYPE_COLORS[t] for t in pivot_types.columns])
    ax.set_title("Genome composition by interpreted state type", fontsize=11, fontweight="bold")
    ax.set_ylabel("Average fraction of genome", fontsize=9)
    ax.set_xlabel("Method", fontsize=9)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize="small", title="Type")
    ax.set_xticks(np.arange(len(pivot_types)))
    ax.set_xticklabels(pivot_types.index, rotation=45, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    plt.tight_layout()
    plt.savefig("out/summary_plots/interpretation_composition.png", bbox_inches="tight")
    plt.close(fig)

    # Plot 6c. Genome fraction per interpreted type, without Quiescent
    signal = frac_long[frac_long["Type"] != "Quiescent"]
    types_present = [t for t in TYPE_ORDER
                     if t != "Quiescent" and signal.loc[signal["Type"] == t, "Fraction"].sum() > 0]
    plt.figure(figsize=(max(12, 1.7 * len(types_present)), 5))
    ax = plt.gca()
    sns.barplot(data=signal, x="Type", y="Fraction", hue="Method", order=types_present,
                hue_order=METHOD_ORDER, palette=method_palette, ax=ax, capsize=0.05,
                errorbar="se", err_kws={"linewidth": 2.0}, edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(ax, METHOD_ORDER)
    utils.strip_points(ax, data=signal, x="Type", y="Fraction", hue="Method",
                       order=types_present, hue_order=METHOD_ORDER, size=2, alpha=0.5, jitter=0.2)
    ax.set_title("Fraction of genome per interpreted state type",
                 fontsize=10, fontweight="bold")
    ax.set_ylabel("Fraction of genome", fontsize=9)
    ax.set_xlabel("Interpreted state type", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="x", labelsize=8, rotation=30)
    ax.legend(title="Method", fontsize=7, title_fontsize=8, bbox_to_anchor=(1.01, 1),
              loc="upper left", borderaxespad=0)
    plt.tight_layout()
    plt.savefig("out/summary_plots/interpretation_fraction.png", bbox_inches="tight")
    plt.close()

    # Plot 6d. How many states each method spends on every type
    counts = (df_interp.groupby(seg_keys + ["Type"]).size().unstack("Type")
              .reindex(columns=TYPE_ORDER).fillna(0).stack().reset_index()
              .rename(columns={0: "States", "level_3": "Type"}))
    plt.figure(figsize=(max(12, 1.7 * len(TYPE_ORDER)), 5))
    ax = plt.gca()
    sns.barplot(data=counts, x="Type", y="States", hue="Method", order=TYPE_ORDER,
                hue_order=METHOD_ORDER, palette=method_palette, ax=ax, capsize=0.05,
                errorbar="se", err_kws={"linewidth": 2.0}, edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(ax, METHOD_ORDER)
    utils.strip_points(ax, data=counts, x="Type", y="States", hue="Method", order=TYPE_ORDER,
                       hue_order=METHOD_ORDER, size=2, alpha=0.5, jitter=0.2)
    ax.set_title("Number of states per interpreted type", fontsize=10, fontweight="bold")
    ax.set_ylabel("States per segmentation", fontsize=9)
    ax.set_xlabel("Interpreted state type", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="x", labelsize=8, rotation=30)
    ax.legend(title="Method", fontsize=7, title_fontsize=8, bbox_to_anchor=(1.01, 1),
              loc="upper left", borderaxespad=0)
    plt.tight_layout()
    plt.savefig("out/summary_plots/interpretation_states_per_type.png", bbox_inches="tight")
    plt.close()

    # Plot 6e. Interpretability per method: types covered, and how clean the calls are
    summary = (df_interp.groupby(seg_keys)
               .agg(Types=("Type", "nunique"), Margin=("Margin", "mean")).reset_index())
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, col, title, ylabel in [
        (axes[0], "Types", "Distinct state types recovered", "Types per segmentation"),
        (axes[1], "Margin", "Interpretation margin (higher = less mixed marks)", "Mean margin"),
    ]:
        sns.barplot(data=summary, x="Method", y=col, order=METHOD_ORDER, hue="Method",
                    hue_order=METHOD_ORDER, palette=method_palette, dodge=False, ax=ax,
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                    edgecolor="lightgrey", linewidth=1, legend=False)
        utils.hatch_joint(ax, METHOD_ORDER)
        utils.strip_points(ax, data=summary, x="Method", y=col, order=METHOD_ORDER,
                           dodge=False, size=3, alpha=0.6)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_xlabel("")
        ax.set_xticks(np.arange(len(METHOD_ORDER)))
        ax.set_xticklabels(METHOD_ORDER, rotation=45, ha="right", fontsize=8)
        ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    plt.tight_layout()
    plt.savefig("out/summary_plots/interpretation_summary.png", bbox_inches="tight")
    plt.close(fig)

    print("Average fraction of genome per interpreted type:")
    display(pivot_types.round(4))

    show_all([f"interpretation_{ds}.png" for ds in DATASETS] + [
             "interpretation_composition.png", "interpretation_fraction.png",
             "interpretation_states_per_type.png", "interpretation_summary.png"],
             base="out/summary_plots", titles=True)


In [ ]:
# 6f. Genome composition per method, over the interpreted state types.
#
# The raw state names cannot carry this plot: process_sagaconf.sh matched every
# model to the joint model of its own dataset, and each of those numbers its
# states arbitrarily, so state "5" of MCF7 and state "5" of K562 are different
# states and the mean of a state over the datasets means nothing - which is
# what avg_composition.png above shows. The interpreted types are the same
# vocabulary in every dataset, so the composition is read over them instead.
# One bar per method, the joint models hatched; there is no reference bar,
# ref_chromhmm is null for every SAGAconf dataset. Plot 6b above is the
# stacked form of the same numbers.
if not df_interp.empty:
    # Types absent from a segmentation contribute 0, so the bars compare the
    # methods on the same footing, and a type no method ever calls is left out.
    frac_by_type = (df_interp.groupby(["Dataset", "Rep", "Method", "Type"])["Fraction"]
                    .sum().unstack("Type").reindex(columns=TYPE_ORDER).fillna(0.0))
    types_present = [t for t in TYPE_ORDER if frac_by_type[t].sum() > 0]

    # Broken between BREAK_LOW and BREAK_HIGH, as every composition plot of the
    # project is: Quiescent covers most of the genome and flattens the rest of
    # the types on a shared axis.
    utils.broken_bar_plot(
        frac_by_type[types_present].stack().reset_index(name="Fraction"),
        "Type", "Fraction", order=types_present, hue="Method",
        hue_order=METHOD_ORDER, palette=method_palette,
        break_low=BREAK_LOW, break_high=BREAK_HIGH,
        figsize=(max(12, len(types_present) * len(METHOD_ORDER) * 0.3), 6),
        title="Genome composition by interpreted state type",
        xlabel="Interpreted state type", ylabel="Fraction of genome",
        legend=True, points={"size": 2, "alpha": 0.5, "jitter": 0.2},
        path="out/summary_plots/composition_methods.png")


In [ ]:
# 6f. Replicate consistency on interpreted NOQH states only

# The background types the NOQH domain drops live in utils, next to the
# NOQH_STATES the name-based metrics drop.


def interpretation_map():
    """(Dataset, Method, Rep, State) -> interpreted Type."""
    return df_interp.set_index(["Dataset", "Method", "Rep", "State"])["Type"].to_dict()


def type_agreement_pair(side1, side2, interp_map):
    """Two segmentations reduced to interpreted state types.

    Each side is a (dataset, method_key, method_name, rep) tuple. Returns the
    (overlap, lengths1, lengths2) triple that match.agreement_metrics() and
    match.per_state_agreement() take, keyed by interpreted type instead of by
    state, or None when either side is not on disk.

    Reducing to types is also what lets two *different* datasets be compared
    without emission rematching: process_sagaconf.sh matches every model to the
    joint model of its own dataset, so state "13" of MCF7 and state "13" of
    K562 are unrelated names, while the interpreted types are a vocabulary
    every segmentation shares.
    """
    (ds1, key1, name1, rep1), (ds2, key2, name2, rep2) = side1, side2
    p1 = get_method_path(ds1, key1, rep1)
    p2 = get_method_path(ds2, key2, rep2)
    if not p1 or not p2 or not os.path.exists(p1) or not os.path.exists(p2):
        return None

    s1, s2 = match.load_bed(p1), match.load_bed(p2)
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    # pair_overlap() puts its *work* state first, so side2 goes in as the
    # reference and the keys come out (side1_state, side2_state).
    overlap = match.pair_overlap(s2, s1)

    # Interpreted type of every state, per side. A state the interpretation
    # never reached counts as background rather than as a type of its own.
    m1 = {s: interp_map.get((ds1, name1, rep1, str(s)), QUIESCENT) for s in l1}
    m2 = {s: interp_map.get((ds2, name2, rep2, str(s)), QUIESCENT) for s in l2}

    type_overlap = defaultdict(float)
    for (st1, st2), val in overlap.items():
        type_overlap[(m1[st1], m2[st2])] += val

    type_l1, type_l2 = defaultdict(float), defaultdict(float)
    for st1, val in l1.items():
        type_l1[m1[st1]] += val
    for st2, val in l2.items():
        type_l2[m2[st2]] += val

    return type_overlap, type_l1, type_l2


def type_agreement_inputs(ds, method_key, method_name, interp_map):
    """rep1 / rep2 of one segmentation, reduced to interpreted state types."""
    return type_agreement_pair((ds, method_key, method_name, "rep1"),
                               (ds, method_key, method_name, "rep2"),
                               interp_map)


def type_agreement_rows(pairs, desc, interp_map):
    """DataFrame of the NOQH agreement of every (method_name, label, side1,
    side2) pair, with the background types of NOQH_TYPES dropped."""
    rows = []
    for method_name, label, side1, side2 in tqdm(pairs, desc=desc):
        inputs = type_agreement_pair(side1, side2, interp_map)
        if inputs is None:
            continue
        metrics = match.agreement_metrics(*inputs, exclude=NOQH_TYPES)
        if metrics:
            rows.append({"Method": method_name, "Dataset": label,
                         **{utils.metric_display(m): v for m, v in metrics.items()}})
    return pd.DataFrame(rows)


def datasets_with_replicates():
    """Datasets carrying both replicate folders on disk."""
    return [ds for ds in DATASETS
            if os.path.isdir(f"{ds}/rep1") and os.path.isdir(f"{ds}/rep2")]


def compute_replicate_agreement_noqh():
    """rep1 vs rep2 of the same method, restricted to NOQH states (interpreted)."""
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", JACCARD_DISPLAY, KAPPA_DISPLAY, COSINE_DISPLAY])

    interp_map = interpretation_map()

    rows = []
    for ds in tqdm(datasets_with_replicates(), desc="Replicates NOQH"):
        for method_key, method_name in METHOD_LABELS:
            inputs = type_agreement_inputs(ds, method_key, method_name, interp_map)
            if inputs is None:
                continue
            metrics = match.agreement_metrics(*inputs, exclude=NOQH_TYPES)
            if metrics:
                rows.append({"Method": method_name, "Dataset": ds,
                             **{utils.metric_display(m): v for m, v in metrics.items()}})

    df = pd.DataFrame(rows)
    return df


def compute_replicate_agreement_per_type():
    """rep1 vs rep2 agreement for each interpreted state type.

    One row per type a method actually emits: match.per_state_agreement() leaves
    out the types absent from both replicates instead of scoring them a perfect
    1.0, which is what an empty intersection over an empty union used to give.

    There is no cosine per type: match.per_state_agreement() reports Jaccard
    and kappa only, and a composition vector of a single type has one
    component, so its cosine would be 1.0 by construction.
    """
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", "Type", JACCARD_DISPLAY, KAPPA_DISPLAY])

    interp_map = interpretation_map()

    rows = []
    for ds in tqdm(datasets_with_replicates(), desc="Replicates per type"):
        for method_key, method_name in METHOD_LABELS:
            inputs = type_agreement_inputs(ds, method_key, method_name, interp_map)
            if inputs is None:
                continue
            for t, metrics in match.per_state_agreement(*inputs).items():
                rows.append({"Method": method_name, "Dataset": ds, "Type": t,
                             **{utils.metric_display(m): v for m, v in metrics.items()}})

    df = pd.DataFrame(rows)
    return df


df_rep_noqh = utils.cached_pickle("out/df_rep_noqh.pkl", compute_replicate_agreement_noqh,
                                  label="replicate consistency (NOQH)", valid=nonempty)
df_rep_per_type = utils.cached_pickle("out/df_rep_per_type.pkl", compute_replicate_agreement_per_type,
                                      label="replicate consistency per type", valid=nonempty)

if not df_rep_per_type.empty:
    # Only the types some method emits; an absent type no longer scores 1.0, so
    # forcing the background trio in here would just add empty x positions.
    types_order = [t for t in TYPE_ORDER if t in set(df_rep_per_type["Type"])]
    for metric, metric_key in [(JACCARD_DISPLAY, JACCARD),
                               (KAPPA_DISPLAY, KAPPA)]:
        plt.figure(figsize=(max(12, len(types_order) * 2), 6))
        ax = sns.barplot(data=df_rep_per_type, x="Type", y=metric, hue="Method",
                         order=types_order, hue_order=METHOD_ORDER, palette=method_palette,
                         capsize=0.1, errorbar="se", edgecolor="lightgrey", linewidth=1)
        utils.hatch_joint(ax, METHOD_ORDER)
        utils.strip_points(ax, data=df_rep_per_type, x="Type", y=metric, hue="Method",
                           order=types_order, hue_order=METHOD_ORDER, size=2.5, alpha=0.6, jitter=0.2)
        
        ax.set_title(f"Replicate {metric} per interpreted state type", fontsize=11, fontweight="bold")
        ax.set_ylabel(metric, fontsize=9)
        ax.set_xlabel("Interpreted Type", fontsize=9)
        ax.set_ylim(0 if metric_key == JACCARD else None, 1.05)
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=45, ha="right")
        ax.legend(title="Method", fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
        
        plt.savefig(f"out/summary_plots/interpretation_replicate_{metric_key}_per_type.png", bbox_inches='tight')
        plt.close()

        # Combined across methods
        plt.figure(figsize=(max(12, len(types_order) * 2), 6))
        ax = sns.barplot(data=df_rep_per_type, x="Type", y=metric,
                         order=types_order, color="#4878CF",
                         capsize=0.1, errorbar="se", edgecolor="lightgrey", linewidth=1)
        utils.strip_points(ax, data=df_rep_per_type, x="Type", y=metric,
                           order=types_order, size=2.5, alpha=0.6, jitter=0.2)
        ax.set_title(f"Average Replicate {metric} per interpreted type (combined across methods)", 
                     fontsize=11, fontweight="bold")
        ax.set_ylabel(metric, fontsize=9)
        ax.set_xlabel("Interpreted Type", fontsize=9)
        ax.set_ylim(0 if metric_key == JACCARD else None, 1.05)
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=45, ha="right")
        plt.savefig(f"out/summary_plots/interpretation_replicate_{metric_key}_per_type_combined.png", 
                    bbox_inches='tight')
        plt.close()

if not df_rep_noqh.empty:
    for metric, metric_key in [(JACCARD_DISPLAY, JACCARD),
                               (KAPPA_DISPLAY, KAPPA),
                               (COSINE_DISPLAY, COSINE)]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df_rep_noqh, x='Method', y=metric, order=METHOD_ORDER,
                    palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                    edgecolor="lightgrey", linewidth=1)
        utils.hatch_joint(plt.gca(), METHOD_ORDER)
        utils.strip_points(plt.gca(), data=df_rep_noqh, x='Method', y=metric, order=METHOD_ORDER, size=2)
        plt.title(f"Replicate {metric} (Interpreted NOQH states)", fontsize=11, fontweight="bold")
        plt.ylabel(metric, fontsize=9)
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis='y', alpha=0.3)

        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(METHOD_ORDER):
            vals = df_rep_noqh[df_rep_noqh["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)

        plt.savefig(f"out/summary_plots/interpretation_replicate_{metric_key}_noqh.png", bbox_inches='tight')
        plt.close()


## 6g. Cross-sample and individual vs joint concordance on interpreted NOQH states


In [ ]:
# The NOQH counterparts of sections 4 and 5 above: the same two comparisons,
# with the background types of NOQH_TYPES dropped so the agreement is read over
# the states the callers actually disagree about rather than over Quiescent.
#
# summary.ipynb scores the composition (cosine) axes on NOQH alone, because the
# Full cosine saturates at 1 as soon as one state dominates both sides; without
# these two caches SAGAconf would contribute to the replicate composition axis
# only, and drop off the cross-sample and individual/joint ones.
def compute_cross_sample_agreement_noqh():
    """Every pair of cell lines, per method, NOQH types only. rep1 only, so the
    replicate variation of 6f does not enter twice."""
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", JACCARD_DISPLAY,
                                     KAPPA_DISPLAY, COSINE_DISPLAY])
    datasets = list(DATASETS)
    pairs = [(method_name, f"{ds_a}/{ds_b}",
              (ds_a, method_key, method_name, "rep1"),
              (ds_b, method_key, method_name, "rep1"))
             for method_key, method_name in METHOD_LABELS
             for i, ds_a in enumerate(datasets)
             for ds_b in datasets[i + 1:]]
    return type_agreement_rows(pairs, "Cross-sample NOQH", interpretation_map())


def compute_joint_indiv_agreement_noqh():
    """Each peak caller on its own dataset vs segmented jointly, NOQH types only."""
    if df_interp.empty:
        return pd.DataFrame(columns=["Method", "Dataset", JACCARD_DISPLAY,
                                     KAPPA_DISPLAY, COSINE_DISPLAY])
    pairs = [(name, ds,
              (ds, indiv_key, name, "rep1"),
              (ds, joint_key, utils.display_name(joint_key), "rep1"))
             for ds in DATASETS
             for name, indiv_key, joint_key in COMP_PAIRS]
    return type_agreement_rows(pairs, "Joint vs Individual NOQH",
                               interpretation_map())


df_cross_noqh = utils.cached_pickle("out/df_cross_sample_noqh.pkl",
                                    compute_cross_sample_agreement_noqh,
                                    label="cross-sample concordance (NOQH)",
                                    valid=nonempty)
df_ji_noqh = utils.cached_pickle("out/df_ji_noqh.pkl",
                                 compute_joint_indiv_agreement_noqh,
                                 label="joint vs individual consistency (NOQH)",
                                 valid=nonempty)

# The joint models have no individual-vs-joint bar of their own, so that panel
# is drawn over the four callers only, the way section 5 draws it.
JI_ORDER = [utils.display_name(m) for m in
            [CHROMHMM_DEFAULT, KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI]]

for df, order, stem, title_prefix in [
        (df_cross_noqh, METHOD_ORDER, "interpretation_cross_sample", "Cross-sample"),
        (df_ji_noqh, JI_ORDER, "interpretation_joint_indiv", "Individual vs joint")]:
    if df.empty:
        continue
    for metric, metric_key in [(JACCARD_DISPLAY, JACCARD),
                               (KAPPA_DISPLAY, KAPPA),
                               (COSINE_DISPLAY, COSINE)]:
        plt.figure(figsize=(8, 6))
        sns.barplot(data=df, x='Method', y=metric, order=order,
                    palette=method_palette, hue='Method', hue_order=order, dodge=False,
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                    edgecolor="lightgrey", linewidth=1)
        utils.hatch_joint(plt.gca(), order)
        utils.strip_points(plt.gca(), data=df, x='Method', y=metric, order=order, size=2)
        plt.title(f"{title_prefix} {metric} (Interpreted NOQH states)",
                  fontsize=11, fontweight="bold")
        plt.ylabel(metric, fontsize=9)
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis='y', alpha=0.3)
        yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
        for i, method in enumerate(order):
            vals = df[df["Method"] == method][metric]
            if vals.empty: continue
            m, s = vals.mean(), vals.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), vals.max())
            plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
        plt.savefig(f"out/summary_plots/{stem}_{metric_key}_{NOQH}.png", bbox_inches='tight')
        plt.close()


## 6h. Transition matrix entropy on interpreted NOQH states

In [ ]:
# The NOQH counterpart of section 2. analyze.save_transition_entropy() drops
# utils.NOQH_STATES by name, which needs the reference state names; a SAGAconf
# state is a number matched to the joint model of its own dataset, so
# run_analysis() runs with skip_noqh=True and only the FULL entropy is on disk.
# The background here is the states interpreted as Quiescent or heterochromatin
# (NOQH_TYPES) - the same domain as the NOQH agreement caches above - and the
# other states keep their own granularity, as they do in the ENCODE NOQH
# entropy.


def compute_entropy_noqh():
    """Transition matrix entropy with the interpreted background dropped."""
    if df_interp.empty:
        return pd.DataFrame(columns=["segmentation", "total_entropy", "n_states",
                                     "Dataset", "Replicate", "Method"])
    background = {key: set(group.loc[group["Type"].isin(NOQH_TYPES),
                                     "State"].astype(str))
                  for key, group in df_interp.groupby(["Dataset", "Method", "Rep"])}
    tasks = [(ds, key, name, rep, path)
             for ds in DATASETS
             for key, name, rep, path in dataset_segmentations(ds)]
    rows = []
    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() // 2)) as executor:
        futures = {}
        for ds, key, name, rep, path in tasks:
            futures[executor.submit(analyze.segmentation_entropy, path,
                                    method_bin_size(key),
                                    background.get((ds, name, rep), set()))] = \
                (ds, key, name, rep)
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc="Entropy NOQH"):
            ds, key, name, rep = futures[fut]
            entropy, n_states = fut.result()
            rows.append({"segmentation": f"{key}_{rep}", "total_entropy": entropy,
                         "n_states": n_states, "Dataset": ds, "Replicate": rep,
                         "Method": name})
    return pd.DataFrame(rows)


df_entropy_noqh = utils.cached_pickle(
    "out/df_entropy_noqh.pkl", compute_entropy_noqh,
    label="transition matrix entropy (NOQH)", valid=nonempty)

if not df_entropy_noqh.empty:
    print(df_entropy_noqh.groupby("Method")[["total_entropy", "n_states"]]
          .mean().round(3).to_string())

    plt.figure(figsize=(12, 5))
    sns.barplot(data=df_entropy_noqh, x='Method', y='total_entropy', order=METHOD_ORDER,
                palette=method_palette, hue='Method', hue_order=METHOD_ORDER, dodge=False,
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1)
    utils.hatch_joint(plt.gca(), METHOD_ORDER)
    utils.strip_points(plt.gca(), data=df_entropy_noqh, x='Method', y='total_entropy',
                       order=METHOD_ORDER, size=2)
    plt.title(f"Transition matrix entropy per method "
              f"({utils.domain_display(NOQH)}, interpreted background)",
              fontsize=11, fontweight="bold")
    plt.ylabel("Entropy (bits)", fontsize=9)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.grid(axis='y', alpha=0.3)
    yrange = plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0]
    for i, method in enumerate(METHOD_ORDER):
        vals = df_entropy_noqh[df_entropy_noqh["Method"] == method]["total_entropy"]
        if vals.empty: continue
        m, s = vals.mean(), vals.sem()
        if pd.isna(m): continue
        top = max(m + (s if not pd.isna(s) else 0), vals.max())
        plt.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
    plt.savefig("out/summary_plots/entropy_noqh.png", bbox_inches='tight')
    plt.close()


## Show

In [ ]:
show_all(["n_peaks.png", "peak_length.png", "n_segments.png", "entropy.png",
              "entropy_noqh.png",
              "avg_composition.png", "avg_composition_stacked.png",
              "composition_methods.png",
              "avg_mean_length.png", "avg_median_length.png",
              "replicate_jaccard.png", "replicate_kappa.png", "replicate_cosine.png",
              "cross_sample_jaccard.png", "cross_sample_kappa.png",
              f"cross_sample_{COSINE}.png",
              "joint_indiv_jaccard.png", "joint_indiv_kappa.png", "joint_indiv_cosine.png",
              f"interpretation_replicate_{JACCARD}_{NOQH}.png",
              f"interpretation_replicate_{KAPPA}_{NOQH}.png",
              f"interpretation_replicate_{COSINE}_{NOQH}.png",
              f"interpretation_replicate_{JACCARD}_per_type.png",
              f"interpretation_replicate_{KAPPA}_per_type.png",
              f"interpretation_replicate_{JACCARD}_per_type_combined.png",
              f"interpretation_replicate_{KAPPA}_per_type_combined.png",
              f"interpretation_cross_sample_{JACCARD}_{NOQH}.png",
              f"interpretation_cross_sample_{KAPPA}_{NOQH}.png",
              f"interpretation_cross_sample_{COSINE}_{NOQH}.png",
              f"interpretation_joint_indiv_{JACCARD}_{NOQH}.png",
              f"interpretation_joint_indiv_{KAPPA}_{NOQH}.png",
              f"interpretation_joint_indiv_{COSINE}_{NOQH}.png"],
             base="out/summary_plots", titles=True)

# State composition per dataset, one plot per method.
show_all([f"composition_{m.lower().replace(' ', '_')}.png" for m in METHOD_ORDER],
         base="out/summary_plots", titles=True)


In [ ]:

# # Display per-dataset results
# for ds in DATASETS:
#     header(f"Dataset: {ds}", 3)
#     for method_key, method_name in METHOD_LABELS:
#         show_group(f"Method: {method_name}", [
#             (method_plot(ds, method_key, "bin_emissions/state_emissions.png",
#                          MATCH_METHOD), f"{method_name}: binarized emissions"),
#             (method_plot(ds, method_key, "enrichment/enrichment.png",
#                          MATCH_METHOD), f"{method_name}: functional enrichment"),
#         ], level=4)